# Traffic Signal OpenEnv: LLM-Based Reinforcement Learning Pipeline

This notebook implements a hardened RL fine-tuning pipeline for hierarchical traffic control.

In [ ]:
import subprocess, sys

print("Installing core dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "trl", "transformers", "accelerate", "peft",
    "bitsandbytes", "datasets", "requests",
    "wandb", "huggingface_hub", "numpy", "matplotlib"
], check=True)
print("Core dependencies installed.")

print("Installing Unsloth...")
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)
print("Unsloth installed.")
print("All dependencies ready.")



In [ ]:
import os, gc, json, re, time, random, requests
import numpy as np
import torch
import wandb
from datasets import Dataset
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOTrainer, GRPOConfig



In [ ]:
PatchFastRL("GRPO", FastLanguageModel)

ENV_URL = "https://guuru-dev-traffic-signal-openenv-2.hf.space"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

# Ensure these are set in kernel
assert os.getenv("HF_TOKEN"), "HF_TOKEN not set"
assert os.getenv("ENV_URL") or ENV_URL, "ENV_URL not set"

try:
    wandb.init(project="traffic-signal-rl", name="full-run-1")
except Exception as e:
    print(f"WandB init failed (non-fatal): {e}")
    wandb = None

r = requests.get(f"{ENV_URL}/health", timeout=30)
print("health:", r.status_code, r.text)
assert r.status_code == 200

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision config -> bf16={use_bf16}, fp16={use_fp16}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=torch.bfloat16 if use_bf16 else None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.generation_config.do_sample = True
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.9
print("Model loaded.")



In [ ]:
PROMPT_BANK = [
    "Control traffic at a 4-way intersection. Output JSON with local_actions and central_action.",
    "Minimize congestion and waiting time. Return valid JSON actions.",
    "Optimize traffic flow under heavy load conditions.",
    "Handle emergency vehicles with priority while keeping flow balanced.",
    "Reduce queue lengths across all intersections.",
    "Prevent spillback and gridlock using coordinated actions.",
    "Balance throughput and fairness across directions.",
    "Respond to dynamic traffic demand changes efficiently.",
    "Avoid starvation of any direction while optimizing flow.",
    "Maximize throughput without increasing waiting time.",
    "Handle congestion recovery after traffic spikes.",
    "Ensure smooth transitions between traffic phases.",
    "Use central coordination to improve global performance.",
    "Prioritize critical lanes under pressure.",
    "Optimize intersection switching intelligently.",
    "Maintain stability while improving efficiency.",
    "Avoid oscillatory switching behavior.",
    "Keep traffic flowing evenly across all directions.",
    "Adapt to changing traffic density dynamically.",
    "Prevent long queues from forming.",
]

train_dataset = Dataset.from_dict({"prompt": PROMPT_BANK * 2})
collected_data = []

def safe_post(url, payload, retries=16, timeout=60):
    for attempt in range(retries):
        try:
            rr = requests.post(url, json=payload, timeout=timeout)
            if rr.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(f"HTTP {rr.status_code}. Waiting {wait:.1f}s ({attempt+1}/{retries})")
                time.sleep(wait)
                continue
            rr.raise_for_status()
            return rr
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            time.sleep(min(30, 2 + attempt))
    raise RuntimeError(f"Failed after {retries} retries: {url}")

def parse_action(completion):
    valid = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
    base = {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}
    try:
        action = json.loads(completion)
        if isinstance(action, dict):
            raw_local = action.get("local_actions", {})
            clean = {}
            for k in ("NW", "NE", "SW", "SE"):
                v = str(raw_local.get(k, "KEEP")).upper().strip() if isinstance(raw_local, dict) else "KEEP"
                clean[k] = v if v in valid else "KEEP"
            return {"local_actions": clean, "central_action": {}}, False
    except Exception:
        pass
    try:
        m = re.search(r"\{.*\}", completion, re.DOTALL)
        if m:
            parsed, _ = parse_action(m.group())
            return parsed, True
    except Exception:
        pass
    return {"local_actions": base, "central_action": {}}, True

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for episode, (_, completion) in enumerate(zip(prompts, completions), start=1):
        task_id = "medium_dynamic" if episode < 20 else "hard_multi"
        safe_post(f"{ENV_URL}/reset", {"task_id": task_id, "central_enabled": True})

        action, is_hallucinated = parse_action(completion)
        episode_reward, done, step_count = 0.0, False, 0
        info, latency_ms = {}, 0.0

        while not done and step_count < 100:
            t0 = time.time()
            try:
                step_res = safe_post(f"{ENV_URL}/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    action = {"local_actions": {"NW":"KEEP","NE":"KEEP","SW":"KEEP","SE":"KEEP"}, "central_action": {}}
                    step_res = safe_post(f"{ENV_URL}/step", action)
                else:
                    raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {})
            episode_reward += float(data.get("reward", 0.0))
            done = data.get("done", False)
            step_count += 1
            time.sleep(0.05)

        if is_hallucinated:
            episode_reward *= 0.5

        collected_data.append({
            "episode_reward": episode_reward,
            "mean_queue": info.get("mean_queue", 0.0),
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "step_count": step_count,
            "is_hallucinated": float(is_hallucinated),
        })

        if wandb:
            wandb.log({
                "episode_reward": episode_reward,
                "mean_queue": info.get("mean_queue", 0.0),
                "final_score": info.get("final_score", 0.0),
                "throughput": info.get("throughput", 0.0),
                "step_count": step_count,
                "step_latency_ms": latency_ms,
                "is_hallucinated": float(is_hallucinated),
            })

        if episode % 5 == 0:
            print(f"[DEBUG] reward={episode_reward:.4f} | parsed_action={action}")

        rewards.append(episode_reward)
        time.sleep(0.2)

    return rewards

training_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    max_steps=80,
    max_prompt_length=512,
    max_completion_length=64,
    num_generations=4,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="wandb",
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)



In [ ]:
torch.cuda.empty_cache()
gc.collect()
print("Trainer ready. Launching training...")

checkpoint_paths = []
if os.path.exists("./outputs"):
    checkpoint_paths = [
        os.path.join("./outputs", n)
        for n in os.listdir("./outputs")
        if n.startswith("checkpoint-") and os.path.isdir(os.path.join("./outputs", n))
    ]
checkpoint_paths = sorted(checkpoint_paths, key=lambda p: int(p.rsplit("-", 1)[-1]))

if checkpoint_paths:
    latest_checkpoint = checkpoint_paths[-1]
    print("Resuming from:", latest_checkpoint)
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("Starting fresh run")
    trainer.train()

print("\nTraining complete.")
model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")



In [ ]:
if wandb:
    wandb.finish()
print("WandB run complete.")



In [ ]:
import sys
from huggingface_hub import HfApi

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Add it as a Space Secret or export it.")

api = HfApi(token=HF_TOKEN)

assert os.path.exists("outputs/traffic-lora"), "outputs/traffic-lora not found."
print("Model weights found:", os.listdir("outputs/traffic-lora"))

api.create_repo("Guuru-DEV/traffic-signal-agent", repo_type="model", exist_ok=True)
api.upload_folder("outputs/traffic-lora", repo_id="Guuru-DEV/traffic-signal-agent", repo_type="model")
print("Model saved to: https://huggingface.co/Guuru-DEV/traffic-signal-agent")

os.makedirs("plots", exist_ok=True)
try:
    sys.path.insert(0, ".")
    from env.metrics_exporter import generate_training_plots
    if "collected_data" not in globals():
        raise RuntimeError("collected_data not found.")
    generate_training_plots(collected_data, "plots")
    print("Plots generated in plots/")
except Exception as e:
    print(f"Plot generation skipped (non-fatal): {e}")

try:
    if os.path.exists("plots") and len(os.listdir("plots")) > 0:
        api.create_repo("Guuru-DEV/traffic-signal-plots", repo_type="dataset", exist_ok=True)
        api.upload_folder("plots", repo_id="Guuru-DEV/traffic-signal-plots", repo_type="dataset")
        print("Plots saved to: https://huggingface.co/datasets/Guuru-DEV/traffic-signal-plots")
    else:
        print("No plots found — skipping upload.")
except Exception as e:
    print(f"Plot upload skipped (non-fatal): {e}")

